# Phase 4a — Zero-Shot Cross-Modality DR Grading of OLIVES

**MSc dissertation — cross-modality diabetic retinopathy.**

This notebook is a **thin orchestrator**; all logic lives in
`src/analysis/zero_shot_grading.py` and reuses the Phase-3a TTA engine
(`src/analysis/uncertainty.py`) **unchanged**.

**What zero-shot cross-modality grading means.** The DR grader was trained on
EyePACS **colour** fundus. OLIVES is **near-infrared** fundus — a different
imaging modality — and has **no DR grade of its own**. We apply the frozen model
to all 3,142 OLIVES images to predict a grade + uncertainty per image, using the
exact same TTA settings as EyePACS (N=30, full augmentation) so the two are
directly comparable.

**This is the prediction step, not validation.** OLIVES has no ground-truth DR
grade; Phase 4b validates these predictions indirectly against OLIVES's clinical
labels. The **key early signal** is the EyePACS-vs-OLIVES confidence comparison:
if the model is markedly less confident (higher entropy) on near-IR, that is it
appropriately flagging out-of-modality input — an honest behaviour to report,
not a failure. A collapsed grade distribution is likewise a legitimate finding.

Use a **GPU** runtime with Drive mounted.

## 1. Setup & Drive mount

Mount Drive, restore the repo, `cd` in, load the Phase 4a config, pick the device.

In [ ]:
# Mount Drive and restore the repo.
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

REPO_DIR = "/content/dr-dissertation"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main
%cd {REPO_DIR}

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU — set Runtime → Change runtime type → GPU")

In [ ]:
%cd /content/dr-dissertation

In [ ]:
# Load config and pick the device (GPU strongly preferred for the TTA passes).
import torch
from src.utils.config import load_config

cfg = load_config('configs/zero_shot.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device != 'cuda':
    print('WARNING: no GPU — TTA over 3,142 images will be slow.')
print('checkpoint:', cfg.checkpoint)
print('eyepacs cache:', cfg.eyepacs_tta_cache)

## 2. STEP 1 — introspection (mandatory, before anything else)

Confirm the **actual** OLIVES loader fields so the code is wired to real keys.
We print: the total image count and the raw-dict keys; one batch's fields
(image tensor shape/dtype; `dr_labels` — expected to be the **-1 sentinel**,
i.e. no DR grade; the clinical/biomarker mask flags; patient_ids); a sample
`metadata` entry (for `visit`); and the EyePACS TTA cache keys used in the
confidence comparison.

In [ ]:
# (a) OLIVES loader + raw dict fields (total count, tier flags, no DR grade).
from src.analysis import uncertainty, zero_shot_grading

data_cfg = uncertainty.read_merged_config(cfg.checkpoint)
loader, olives = zero_shot_grading.build_olives_loader(data_cfg, int(cfg.tta.batch_size))
print('total images:', tuple(olives['images'].shape))
print('raw olives dict keys:', list(olives.keys()))
print('has_clinical / has_biomarkers present:',
      'has_clinical' in olives, 'has_biomarkers' in olives)
print('eye_ids present:', 'eye_ids' in olives, '| metadata[0]:', olives['metadata'][0])

In [ ]:
# (b) One batch: confirm image shapes and that dr_labels is the -1 sentinel.
batch = next(iter(loader))
print('images:', tuple(batch['images'].shape), batch['images'].dtype)
print('dr_labels unique (expect [-1] = OLIVES has no DR grade):',
      batch['dr_labels'].unique().tolist())
print('dataset ids unique (expect [1] = OLIVES):', batch['dataset'].unique().tolist())
print('carries bcva/cst/has_clinical/has_biomarkers/patient_ids:',
      all(k in batch for k in ['bcva', 'cst', 'has_clinical', 'has_biomarkers', 'patient_ids']))

In [ ]:
# (c) EyePACS TTA cache keys (from Phase 3a) — used for the confidence comparison.
eye = torch.load(cfg.eyepacs_tta_cache, map_location='cpu', weights_only=False)
print('eyepacs cache keys:', [k for k in eye.keys() if k != 'meta'])
print('eyepacs n images:', eye['confidence_maxprob'].shape)

## 3. Run zero-shot grading

`run_zero_shot` loads the frozen model, runs the TTA engine over all OLIVES
images (30 augmented + 1 clean pass each; cached to Drive so re-runs are
instant), assembles the per-image prediction table, and writes
`features/tta_olives.pt` + `reports/olives_dr_predictions.csv` — the direct input
to Phase 4b. It also computes the predicted-grade distribution and the
EyePACS-vs-OLIVES confidence/entropy comparison, and writes the report + figures.

In [ ]:
# Run the full pipeline (idempotent: the OLIVES TTA cache is reused if present).
summary = zero_shot_grading.run_zero_shot(cfg, device)
print('\ngrade distribution:', summary['grade_distribution'])
if summary['comparison'] is not None:
    print('confidence diff (OLIVES - EyePACS):',
          round(summary['comparison']['confidence_mean_diff_olives_minus_eyepacs'], 4))

## 4. Report

The cautious interpretation: predicted-grade distribution (flagging collapse),
OLIVES uncertainty, the EyePACS-vs-OLIVES confidence signal, and an explicit
reminder that these grades are **unvalidated until Phase 4b**.

In [ ]:
# Render the markdown report inline.
from pathlib import Path
from IPython.display import Markdown, display

report_dir = Path(str(cfg.report_dir))
display(Markdown((report_dir / 'phase4a_report.md').read_text(encoding='utf-8')))

## 5. Figures

1. **OLIVES predicted-grade distribution** — is it spread or collapsed?
2. **Confidence: EyePACS vs OLIVES** — the key figure: is the model less confident
   on out-of-modality near-IR?
3. **Entropy: EyePACS vs OLIVES** — higher OLIVES entropy corroborates (2).
4. **Example OLIVES predictions** — near-IR images with predicted grade +
   confidence (+ BCVA/CST context); clearly labelled **no ground-truth grade**.

In [ ]:
# Display each saved PNG inline (some are skipped if the EyePACS cache is absent).
from IPython.display import Image, display

figures_dir = Path(str(cfg.figures_dir))
for stem in [
    'phase4a_fig1_grade_distribution',
    'phase4a_fig2_confidence_compare',
    'phase4a_fig3_entropy_compare',
    'phase4a_fig4_examples',
]:
    png = figures_dir / f'{stem}.png'
    if png.exists():
        display(Image(filename=str(png)))
    else:
        print('(missing)', png.name)

## 6. Output manifest

List everything Phase 4a wrote to Drive: the OLIVES TTA cache + prediction table
(the Phase 4b inputs), the summary JSON + report, and the figures (PNG + PDF).

In [ ]:
# Confirm the artefacts on Drive.
features_dir = Path(str(cfg.features_dir))
print('Phase 4b inputs:')
for p in [features_dir / 'tta_olives.pt', report_dir / 'olives_dr_predictions.csv']:
    print('  ', p.name, '(exists:', p.exists(), ')')
print('reports:')
for p in sorted(report_dir.glob('phase4a*')):
    print('  ', p.name)
print('figures:')
for p in sorted(figures_dir.glob('phase4a*')):
    print('  ', p.name)